In [ ]:
import os
import pandas as pd
from datasets import load_dataset

# 1. 强制连接国内镜像站
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

def get_refined_mrc_seeds(num_seeds=100):
    print("正在从镜像站加载 MRC 数据库...")
    dataset = load_dataset("StephanAkkerman/MRC-psycholinguistic-database", split="train")
    df = pd.DataFrame(dataset)

    # 2. 匹配你打印出的真实列名
    # 长度: 'Number of Letters'
    # 音节: 'Number of Syllables'
    # 词频: 'KF Written Frequency'
    # 具象性: 'Concreteness'
    # 熟悉度: 'Familiarity'
    
    print("开始按照心理语言学指标筛选...")
    
    refined_df = df[
        (df['Word'].notna()) &
        (df['Word'].str.isalpha()) &           
        (df['Number of Letters'].between(4, 6)) &      
        (df['Number of Syllables'] == 2) &             
        (df['KF Written Frequency'].between(10, 100)) &
        (df['Concreteness'] > 400) &           
        (df['Familiarity'] > 400)              
    ].copy()

    if len(refined_df) < num_seeds:
        print(f"⚠️ 提示：符合条件的种子共有 {len(refined_df)} 个。")
        return refined_df
    
    # 随机抽样 100 个
    return refined_df.sample(num_seeds, random_state=42)

try:
    seeds_df = get_refined_mrc_seeds(100)
    
    # 3. 整理输出格式，只保留咱们研究关心的维度
    display_cols = ['Word', 'Number of Letters', 'Number of Syllables', 'KF Written Frequency', 'Concreteness']
    
    print("\n--- ✅ 成功筛选 100 个黄金种子词 (预览) ---")
    print(seeds_df[display_cols].head(10))
    
    # 保存结果
    seeds_df.to_csv('dic_data\mrc_seeds_100.csv', index=False, encoding='utf-8')
    print(f"\n🚀 任务完成！100个词已存入: {os.getcwd()}\dic_data\mrc_seeds_100.csv")

except Exception as e:
    print(f"❌ 运行依然出错: {e}")
    print("请检查拼写是否与数据集列名完全一致。")

正在从镜像站加载 MRC 数据库...
开始按照心理语言学指标筛选...

--- ✅ 成功筛选 100 个黄金种子词 (预览) ---
          Word  Number of Letters  Number of Syllables  KF Written Frequency  \
105299  PRISON                  6                    2                    42   
96053   PALACE                  6                    2                    38   
88214    MOTOR                  5                    2                    56   
121903  SILVER                  6                    2                    29   
19517   CANCER                  6                    2                    25   
145184  VICTIM                  6                    2                    27   
13828   BISHOP                  6                    2                    18   
43611   EATING                  6                    2                    32   
147287  WEAPON                  6                    2                    42   
93149    ONION                  5                    2                    15   

        Concreteness  
105299           570  
9605

In [ ]:
data = pd.read_csv('dic_data\mrc_seeds_100.csv')
data.head()